# Phase 5.1: Simple 1D-CNN for Binary Stress Classification

Trains a small custom CNN with explicit temporal -> spatial -> classify structure. Evaluates on the held out 4-subject test set defined in Phase 5.0.

**Goal**: a working DL baseline to compare against Phase 4B's traditional ML results (~58% binary accuracy with LOSO-CV).

**Result**: Reports both accuracy and macro-F1, plus a majority-class baseline. Because the test set is ~40% Stress, raw accuracy alone is misleading.

In [1]:

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
 
from phase5_utils import (
    load_phase2_data, make_subject_split, make_loaders,
    train_model, evaluate_model, count_parameters,
    PHASE5_DIR, SEED,
)
 
# Reproducibility — same seed everywhere
torch.manual_seed(SEED)
np.random.seed(SEED)

# Architecture

A small custom CNN that explicitly shows the temporal → spatial → classify pattern. Designed to be readable and CPU-friendly.

- **Input**:  (B, 1, 32, 640) [batch, 1 image-channel, 32 EEG ch, 640 samples]
- **Output**: (B, n_classes) [logits]

* *Block 1:* temporal conv     → learns 8 different time-domain filters
* *Block 2:* spatial conv      → collapses 32 EEG channels into 16 mixtures
* *Block 3:* temporal conv (2) → learns slower temporal abstractions
* *Out:*    flatten + linear   → class logits

In [3]:
class SimpleEEGCNN(nn.Module):
 
    # Input:  (B, 1, 32, 640) | batch, 1 image-channel, 32 EEG ch, 640 samples
    # Output: (B, n_classes)  | logits
 
    def __init__(self, n_classes=2, n_temporal=8, n_spatial=16, dropout=0.5):
        super().__init__()

        """
        ---- Block 1: temporal convolution ----
        Kernel (1, 25) = 25 samples ≈ 195 ms at 128 Hz.
        Each output filter learns a temporal pattern across time, independent
        for every EEG channel (channels not mixed yet).
        bias=False because BatchNorm right after makes bias redundant.
        """

        self.temporal_conv = nn.Conv2d(
            in_channels=1, out_channels=n_temporal,
            kernel_size=(1, 25), padding=(0, 12), bias=False,
        )
        self.bn1 = nn.BatchNorm2d(n_temporal)

        """
        ---- Block 2: spatial convolution ----
        Kernel (32, 1) uses ALL 32 EEG channels at one time step. Each of the
        n_spatial output filters learns a different channel mixture
        (a "virtual electrode"). Output height collapses from 32 to 1.
        """

        self.spatial_conv = nn.Conv2d(
            in_channels=n_temporal, out_channels=n_spatial,
            kernel_size=(32, 1), bias=False,
        )
        self.bn2 = nn.BatchNorm2d(n_spatial)
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))   # time 640 → 160
        self.drop1 = nn.Dropout(dropout)

        """
        ---- Block 3: deeper temporal convolution ----
        Operates on the (n_spatial, 1, 160) feature maps. Captures slower /
        more abstract patterns in the spatially-mixed signal.
        """

        self.temporal_conv2 = nn.Conv2d(
            in_channels=n_spatial, out_channels=n_spatial,
            kernel_size=(1, 13), padding=(0, 6), bias=False,
        )
        self.bn3 = nn.BatchNorm2d(n_spatial)
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))   # time 160 → 20
        self.drop2 = nn.Dropout(dropout)

        """
        ---- Classifier ----
        After pooling, feature map is (n_spatial, 1, 20). Flatten = 320 dims.
        """

        self.classifier = nn.Linear(n_spatial * 20, n_classes)

    def forward(self, x):
        # x: (B, 1, 32, 640)

        # ---- Block 1 ----
        x = self.temporal_conv(x)    # (B, 8, 32, 640)
        x = self.bn1(x)
        """
        NOTE: no activation here. Letting the spatial conv operate on raw
        (linear) temporal features is a design choice from EEGNet & Shallow
        ConvNet. It lets the spatial filters learn the best channel mixture
        of the linear temporal features, rather than of already-nonlinear ones.
        """
        
        # ---- Block 2 ----
        x = self.spatial_conv(x)     # (B, 16, 1, 640)
        x = self.bn2(x)
        x = F.elu(x)                 # ELU: smoother than ReLU, suits EEG sign symmetry
        x = self.pool1(x)            # (B, 16, 1, 160)
        x = self.drop1(x)

        # ---- Block 3 ----
        x = self.temporal_conv2(x)   # (B, 16, 1, 160)
        x = self.bn3(x)
        x = F.elu(x)
        x = self.pool2(x)            # (B, 16, 1, 20)
        x = self.drop2(x)

        # ---- Classifier ----
        x = x.flatten(1)             # (B, 320)
        x = self.classifier(x)       # (B, n_classes)
        return x

# Main Script

In [5]:
print("PHASE 5.1: SIMPLE 1D-CNN (binary classification)")
print("=" * 70)
 
 
# 1. Load data + recreate the Phase 5.0 split (deterministic via SEED) --------
print("\n[1/4] Loading data and recreating Phase 5.0 split...")
X, y_binary, subjects = load_phase2_data(verbose=False)
train_idx, val_idx, test_idx, split_info = make_subject_split(subjects, n_train=32, n_val=4, n_test=4, seed=SEED)

train_loader, val_loader, test_loader = make_loaders(X, y_binary, train_idx, val_idx, test_idx, batch_size=64)

print(f"  Train: {train_idx.sum()}  Val: {val_idx.sum()}  Test: {test_idx.sum()}")
print(f"  Test subjects: {split_info['test']}")
 
 
# 2. Build model and sanity-check forward pass --------
print("\n[2/4] Building SimpleEEGCNN...")
device = "cpu"
model = SimpleEEGCNN(n_classes=2).to(device)
n_params = count_parameters(model)
print(f"  Total trainable parameters: {n_params:,}")
 
with torch.no_grad():
    dummy = torch.zeros(2, 1, 32, 640)
    out = model(dummy)
    print(f"  Forward-pass sanity: input {tuple(dummy.shape)} → output {tuple(out.shape)}")
 
 
# 3. Train (early stopping on val_loss, patience=7) --------
print("\n[3/4] Training...")
start = time.time()
best_state, history = train_model(
    model, train_loader, val_loader,
    n_epochs=40, lr=1e-3, weight_decay=1e-4, patience=7,
    device=device, verbose=True,
)

elapsed = time.time() - start
print(f"  Training time: {elapsed:.1f} seconds ({elapsed/60:.1f} min)")
 
# Load the best checkpoint (lowest val_loss epoch) before evaluating
model.load_state_dict(best_state)
 
 
#  4. Evaluate on the held-out test subjects --------
print("\n[4/4] Evaluating on test set (4 unseen subjects)...")
results = evaluate_model(model, test_loader, device=device)
 
# Majority-class baseline on the test set - a "predict majority class always"
# classifier is the minimum bar a real classifier must beat.
n0_test = int((y_binary[test_idx] == 0).sum())
n1_test = int((y_binary[test_idx] == 1).sum())
majority_baseline = max(n0_test, n1_test) / (n0_test + n1_test)
 
print(f"  Test accuracy : {results['accuracy']:.4f}")
print(f"  Test F1 (pos) : {results['f1']:.4f}")
print(f"  Test F1 macro : {results['f1_macro']:.4f}")
print(f"  Confusion matrix:")
print(f"    {results['confusion_matrix']}")
print(f"  ---- Reference points ----")
print(f"  Majority-class baseline on test  : {majority_baseline:.4f}")
print(f"  Phase 4B best (binary RF, LOSO)  : 0.5870")

PHASE 5.1: SIMPLE 1D-CNN (binary classification)

[1/4] Loading data and recreating Phase 5.0 split...
  Train: 1920  Val: 240  Test: 240
  Test subjects: [2, 9, 14, 34]

[2/4] Building SimpleEEGCNN...
  Total trainable parameters: 8,346
  Forward-pass sanity: input (2, 1, 32, 640) → output (2, 2)

[3/4] Training...
  Epoch   1 | train_loss 0.7022 | val_loss 0.6863 | val_acc 0.5500
  Epoch   2 | train_loss 0.6834 | val_loss 0.6856 | val_acc 0.5208
  Epoch   3 | train_loss 0.6731 | val_loss 0.6996 | val_acc 0.5417
  Epoch   4 | train_loss 0.6516 | val_loss 0.7118 | val_acc 0.5292
  Epoch   5 | train_loss 0.6588 | val_loss 0.7338 | val_acc 0.5417
  Epoch   6 | train_loss 0.6376 | val_loss 0.7168 | val_acc 0.5500
  Epoch   7 | train_loss 0.6326 | val_loss 0.7606 | val_acc 0.5250
  Epoch   8 | train_loss 0.6302 | val_loss 0.7591 | val_acc 0.5250
  Epoch   9 | train_loss 0.6160 | val_loss 0.7455 | val_acc 0.5458
  Early stopping at epoch 9 (no improvement for 7 epochs).
  Training time: 28.

# Save Results & Plots

In [6]:
os.makedirs(PHASE5_DIR, exist_ok=True)
out_dir = os.path.join(PHASE5_DIR, "phase5_1_simple_cnn")
os.makedirs(out_dir, exist_ok=True)
 
# 1) Training curves ---------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs = range(1, len(history["train_loss"]) + 1)
 
axes[0].plot(epochs, history["train_loss"], label="Train", color="#5DA5DA", marker="o", markersize=3)
axes[0].plot(epochs, history["val_loss"],   label="Val",   color="#F15854", marker="o", markersize=3)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves"); axes[0].legend(); axes[0].grid(alpha=0.3)
 
axes[1].plot(epochs, history["val_acc"], color="#60BD68", marker="o", markersize=3)
axes[1].axhline(majority_baseline, color="gray", linestyle="--", label=f"Majority baseline ({majority_baseline:.3f})")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1)
 
plt.suptitle(f"Phase 5.1 — Simple 1D-CNN ({n_params:,} params)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150)
plt.close()
 
# 2) Confusion matrix heatmap --------
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(results["confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=["Relaxed", "Stress"], yticklabels=["Relaxed", "Stress"], ax=ax,)

ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Phase 5.1 test confusion matrix (acc={results['accuracy']:.3f}, "f"F1-macro={results['f1_macro']:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
plt.close()
 
# 3) Save numeric summary --------
results_summary = {
    "model": "SimpleEEGCNN",
    "n_params": int(n_params),
    "training_time_sec": float(elapsed),
    "n_epochs_trained": len(history["train_loss"]),
    "test_accuracy": float(results["accuracy"]),
    "test_f1": float(results["f1"]),
    "test_f1_macro": float(results["f1_macro"]),
    "majority_baseline": float(majority_baseline),
    "phase4b_binary_rf_loso": 0.5870,
    "best_val_loss": float(min(history["val_loss"])),
    "best_val_acc": float(max(history["val_acc"])),
    "confusion_matrix": results["confusion_matrix"].tolist(),
    "split_subjects": split_info,
    "hyperparameters": {
        "n_temporal_filters": 8,
        "n_spatial_filters": 16,
        "temporal_kernel_1": 25,
        "temporal_kernel_2": 13,
        "dropout": 0.5,
        "batch_size": 64,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "max_epochs": 40,
        "patience": 7,
    },
}
with open(os.path.join(out_dir, "results.json"), "w") as f:
    json.dump(results_summary, f, indent=2)
 
# 4) Save trained model weights for later analysis --------
torch.save(best_state, os.path.join(out_dir, "simple_cnn_best.pt"))
 
print(f"\nSaved outputs to: {out_dir}")
print("  - training_curves.png")
print("  - confusion_matrix.png")
print("  - results.json")
print("  - simple_cnn_best.pt")
print("=" * 70)
print("Phase 5.1 complete.")
print("=" * 70)


Saved outputs to: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\phase5_1_simple_cnn
  - training_curves.png
  - confusion_matrix.png
  - results.json
  - simple_cnn_best.pt
Phase 5.1 complete.
